In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random

# Fijamos semillas para reproducibilidad básica
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Librerías base importadas correctamente.")

Librerías base importadas correctamente.


## 1. Entorno y Algoritmo

* **Entorno (Taxi-v3):** Consta de 500 estados discretos y 6 acciones posibles. El objetivo es recoger a un pasajero y dejarlo en su destino en el menor número de pasos posible (cada paso penaliza con -1).
* **Q-Learning:** Es un algoritmo de Diferencia Temporal *Off-Policy*. Actualiza la tabla de valores Q utilizando el valor de la **mejor acción posible** en el siguiente estado, sin importar si el agente realmente tomará esa acción o explorará otra:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$

In [2]:
def choose_action(state, Q_table, epsilon, env):
    """Política Epsilon-Greedy"""
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample() # Exploración aleatoria
    else:
        return np.argmax(Q_table[state, :]) # Explotación de la mejor acción

def train_qlearning_basic(env, episodes, alpha, gamma, epsilon_start, epsilon_decay, epsilon_min):
    """Bucle de entrenamiento base para Q-Learning"""
    Q_table = np.zeros((env.observation_space.n, env.action_space.n))
    stats_rewards = []
    stats_lengths = []

    epsilon = epsilon_start

    for ep in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        steps = 0
        done = False

        while not done:
            action = choose_action(state, Q_table, epsilon, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # ── CORAZÓN DE Q-LEARNING (Off-Policy) ──
            # Calculamos el target usando el máximo valor Q del siguiente estado
            best_next_action = np.argmax(Q_table[next_state, :])
            td_target = reward + gamma * Q_table[next_state, best_next_action] * (not done)

            # Actualizamos la tabla Q
            td_error = td_target - Q_table[state, action]
            Q_table[state, action] += alpha * td_error

            state = next_state
            total_reward += reward
            steps += 1

        # Decaimiento del epsilon al terminar el episodio
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        # Guardamos métricas
        stats_rewards.append(total_reward)
        stats_lengths.append(steps)

    return Q_table, stats_rewards, stats_lengths